In [ ]:
!pip install transformers accelerate torch pillow

In [1]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from PIL import Image
import torch

model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
image = Image.open("full.png")

width, height = image.size
scoreboard = image.crop((0, int(height * 0.92), width, int(height * 0.97)))  # stops at 97% instead of 100%

scoreboard.save("scoreboard_crop.png")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": scoreboard},
            {"type": "text", "text": "Extract all text from this image"}
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False)
inputs = processor(text=[text], images=[scoreboard], return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=200)
result = processor.batch_decode(output, skip_special_tokens=True)
print(result[0])

system
You are a helpful assistant.
user
Extract all text from this image

answer: AUS 17-2 1st Inns Toss Aus Smith 0* (3) Khawaja 4 (12) ENG Broad 2-13 (4)


In [ ]:
image = Image.open("full.png")

width, height = image.size
scoreboard = image.crop((0, int(height * 0.92), width, int(height * 0.97)))  # stops at 97% instead of 100%

scoreboard.save("scoreboard_crop.png")  # verify the crop

To delete the gpu cache , These weights gets loaded

In [5]:
!rm -rf /content/captured_frames/

###### this is still the one


In [6]:
import cv2
from PIL import Image
import os
import re

# ── STEP 1: Capture & save cropped frames ──────────────────────────
os.makedirs("captured_frames", exist_ok=True)

cap = cv2.VideoCapture("smith_batting.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps)

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count % frame_interval == 0:
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        width, height = image.size

        scoreboard = image.crop((0, int(height * 0.92), width, int(height * 0.97)))

        timestamp = frame_count / fps
        #save_path = f"captured_frames/frame_{timestamp:.1f}s.png"
        save_path = f"captured_frames/frame_{int(timestamp):04d}s.png"
        scoreboard.save(save_path)

    frame_count += 1

cap.release()
print("All frames captured.")

# ── STEP 2: Natural sorting function ───────────────────────────────
def natural_sort_key(filename):
    match = re.search(r'frame_(\d+\.?\d*)s', filename)
    return float(match.group(1)) if match else 0

# ── STEP 3: Read images in correct order ───────────────────────────
def load_images(folder="captured_frames"):
    image_files = sorted(os.listdir(folder), key=natural_sort_key)

    images = []
    for file in image_files:
        path = os.path.join(folder, file)
        img = Image.open(path)
        images.append(img)
        print(file)  # to verify order

    return images

# Run
images = load_images()

All frames captured.
frame_0000s.png
frame_0001s.png
frame_0002s.png
frame_0003s.png
frame_0004s.png
frame_0005s.png
frame_0006s.png
frame_0007s.png
frame_0008s.png
frame_0009s.png
frame_0010s.png
frame_0011s.png
frame_0012s.png
frame_0013s.png
frame_0014s.png
frame_0015s.png
frame_0016s.png
frame_0017s.png
frame_0018s.png
frame_0019s.png
frame_0020s.png
frame_0021s.png
frame_0022s.png
frame_0023s.png
frame_0024s.png
frame_0025s.png
frame_0026s.png


In [7]:
import torch

def extract_text_from_images(folder="captured_frames"):
    image_files = sorted(os.listdir(folder))

    for img_file in image_files:
        if not img_file.endswith(".png"):
            continue

        img_path = os.path.join(folder, img_file)
        scoreboard = Image.open(img_path).convert("RGB")  # ensure 3-channel

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": scoreboard},
                    {"type": "text", "text": "Extract all text from this image"}
                ],
            }
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[scoreboard], return_tensors="pt").to("cuda")

        input_len = inputs["input_ids"].shape[1]  # track input token length

        with torch.no_grad():  # ← disables grad computation
            output = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False  # greedy decode = faster + deterministic
            )

        # Slice off the input tokens — only decode newly generated tokens
        generated_tokens = output[:, input_len:]
        result = processor.batch_decode(generated_tokens, skip_special_tokens=True)

        print(f"[{img_file}] {result[0]}")

        # Free GPU memory between iterations
        del inputs, output, generated_tokens
        torch.cuda.empty_cache()

extract_text_from_images()

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[frame_0000s.png] Here is the text extracted from the image:

AUS 22-2
1st Inns
Overs 10.4
Smith 4* (12)
Khawaja 5 (13)
ENG Wookes 0-5 (14)
[frame_0001s.png] Here is the text extracted from the image:

AUS 23-2
1st Inns
Overs 11
Smith 5* (13)
Khawaja 5 (14)
ENG Broad 2-13 (5)
[frame_0002s.png] Here is the text extracted from the image:

AUS 23-2 1st Inns Overs 11 Smith 5* (13) Khawaja 5 (14) ENG Broad 2-13 (5)
[frame_0003s.png] AUS 23-2 1st Inns Overs 11 Smith 5* (13) Khawaja 5 (14) ENG Broad 2-13 (5)
[frame_0004s.png] AUS 23-2 Smith Avg by Pace Line Wide 100.0 Channel 49.8 Stumps 60.8 Leg Side 163.7
[frame_0005s.png] AUS 23-2 Smith Avg by Pace Line Wide 100.0 Channel 49.8 Stumps 60.8 Leg Side 163.7
[frame_0006s.png] AUS 23-2 Smith Avg by Pace Line Wide 100.0 Channel 49.8 Stumps 60.8 Leg Side 163.7
[frame_0007s.png] AUS 23-2 1st Inns Overs 11.2 Smith 5* (15) Khawaja 5 (14) ENG Broad 2-13 (52)
[frame_0008s.png] AUS 23-2 1st Inns Overs 11.2 Smith 5* (15) Khawaja 5* (14) ENG Broad 2-13 (5

In [12]:
import torch
import os
import re
from PIL import Image

def fix_bowler_overs(text):
    # Only fix (NN) that comes AFTER a bowler figures pattern like "2-13"
    def fix(m):
        num = m.group(2)
        if "." not in num and len(num) >= 2 and int(num[-1]) <= 5:
            return f"{m.group(1)} ({num[:-1]}.{num[-1]})"
        return m.group(0)
    return re.sub(r'(\d+-\d+)\s+\((\d{2,})\)', fix, text)

def parse_frame(text):
    text = fix_bowler_overs(text)

    score   = re.search(r'(AUS|ENG)\s+(\d+-\d+)', text)
    over    = re.search(r'Overs\s+([\d.]+)', text)
    smith   = re.search(r'Smith\s+(\d+\*?)\s+\((\d+)\)', text)
    khawaja = re.search(r'Khawaja\s+(\d+\*?)\s+\((\d+)\)', text)
    bowler  = re.search(r'(Broad|Wookes|Anderson|Stokes|Wood)\s+(\d+-\d+)\s+\(([\d.]+)\)', text)

    # skip frames missing any key info
    if not score or not smith or not khawaja or not bowler:
        return None

    return {
        "score":   score.group(2),
        "over":    over.group(1) if over else "—",
        "smith":   f"{smith.group(1)} ({smith.group(2)} balls)",
        "khawaja": f"{khawaja.group(1)} ({khawaja.group(2)} balls)",
        "bowler":  bowler.group(1),
        "figures": bowler.group(2),
        "b_overs": bowler.group(3),
    }

def extract_text_from_images(folder="captured_frames"):
    image_files = sorted(os.listdir(folder))
    last_state = None

    for img_file in image_files:
        if not img_file.endswith(".png"):
            continue

        img_path = os.path.join(folder, img_file)
        scoreboard = Image.open(img_path).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": scoreboard},
                    {"type": "text", "text": "Extract all text from this image"}
                ],
            }
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[scoreboard], return_tensors="pt").to("cuda")
        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=200, do_sample=False)

        generated_tokens = output[:, input_len:]
        result = processor.batch_decode(generated_tokens, skip_special_tokens=True)

        parsed = parse_frame(result[0])

        if parsed and parsed != last_state:
            print(f"\n{'─'*45}")
            print(f"  Frame   : {img_file}")
            print(f"  Score   : AUS {parsed['score']}  |  Over : {parsed['over']}")
            print(f"  Smith   : {parsed['smith']}")
            print(f"  Khawaja : {parsed['khawaja']}")
            print(f"  Bowler  : {parsed['bowler']}  {parsed['figures']}  ({parsed['b_overs']} ov)")
            last_state = parsed

        del inputs, output, generated_tokens
        torch.cuda.empty_cache()

extract_text_from_images()


─────────────────────────────────────────────
  Frame   : frame_0000s.png
  Score   : AUS 22-2  |  Over : 10.4
  Smith   : 4* (12 balls)
  Khawaja : 5 (13 balls)
  Bowler  : Wookes  0-5  (1.4 ov)

─────────────────────────────────────────────
  Frame   : frame_0001s.png
  Score   : AUS 23-2  |  Over : 11
  Smith   : 5* (13 balls)
  Khawaja : 5 (14 balls)
  Bowler  : Broad  2-13  (5 ov)

─────────────────────────────────────────────
  Frame   : frame_0007s.png
  Score   : AUS 23-2  |  Over : 11.2
  Smith   : 5* (15 balls)
  Khawaja : 5 (14 balls)
  Bowler  : Broad  2-13  (5.2 ov)

─────────────────────────────────────────────
  Frame   : frame_0008s.png
  Score   : AUS 23-2  |  Over : 11.2
  Smith   : 5* (15 balls)
  Khawaja : 5* (14 balls)
  Bowler  : Broad  2-13  (5.2 ov)

─────────────────────────────────────────────
  Frame   : frame_0014s.png
  Score   : AUS 27-2  |  Over : 11.3
  Smith   : 9* (16 balls)
  Khawaja : 5 (14 balls)
  Bowler  : Broad  2-17  (5.3 ov)

─────────────────

In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()